# Causal Inference in Practice
## Week 6 — Matching & Propensity Scores · Practice Notebook

> **Block II — Adjustment for confounding**
>
> Recreating a randomized comparison by balancing the covariate distributions of treated and control units.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · A confounded world with a KNOWN effect

We simulate an observational study of a job-training-style program. Three covariates — `age`, `educ`, `prior` (earnings history) — drive **both** who gets treated (`D`) and the outcome (`Y`). We build in a **constant individual treatment effect of exactly 2.0**, so the true ATT and ATE are both `2.0` and we can check every estimate against the truth.

Because the treated are systematically different (higher `prior`, etc.), the *naive* treated−control difference will be badly biased.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

TRUE_EFFECT = 2.0          # constant individual effect we build in

def simulate(n=4000):
    age   = RNG.normal(0, 1, n)
    educ  = RNG.normal(0, 1, n)
    prior = 0.6*age - 0.4*educ + RNG.normal(0, 1, n)   # earnings history
    # confounders push the probability of treatment
    logit = 0.8*age - 0.7*educ + 0.6*prior
    ps_true = 1 / (1 + np.exp(-logit))
    D = RNG.binomial(1, ps_true)
    # potential outcomes; Y(1) = Y(0) + TRUE_EFFECT for everyone
    Y0 = 1.0 + 1.2*age - 0.9*educ + 1.1*prior + RNG.normal(0, 1, n)
    Y1 = Y0 + TRUE_EFFECT
    Y  = np.where(D == 1, Y1, Y0)
    return pd.DataFrame({'age': age, 'educ': educ, 'prior': prior,
                         'D': D, 'Y': Y, 'Y0': Y0, 'Y1': Y1})

df = simulate()
# The ATT is computable here because we know both potential outcomes.
TRUE_ATT = (df.loc[df.D==1, 'Y1'] - df.loc[df.D==1, 'Y0']).mean()
print(f'treated n = {int(df.D.sum())},  control n = {int((1-df.D).sum())}')
print(f'TRUE ATT  = {TRUE_ATT:.3f}   (built in as {TRUE_EFFECT})')

### The naive estimate is biased

Just subtract the group means. Because treated units started with higher `prior` (and other confounders), this **overstates** the effect — it credits pre-existing advantages to the program.

In [ ]:
naive = df.loc[df.D==1, 'Y'].mean() - df.loc[df.D==0, 'Y'].mean()
print(f'naive difference = {naive:.3f}')
print(f'true ATT         = {TRUE_ATT:.3f}')
print(f'bias             = {naive - TRUE_ATT:+.3f}   (large!)')
assert naive - TRUE_ATT > 1.0, 'the naive estimate should be badly biased'

## 2 · Fit a propensity score

Estimate `e(X) = P(D=1 | X)` with `LogisticRegression`. Remember: we judge this model by the **balance** it buys, not its accuracy.

In [ ]:
covs = ['age', 'educ', 'prior']
X = df[covs].values
ps_model = LogisticRegression(max_iter=1000).fit(X, df['D'].values)
df['ps'] = ps_model.predict_proba(X)[:, 1]
# logit of the PS — the scale on which we usually set calipers
df['logit_ps'] = np.log(df['ps'] / (1 - df['ps']))
print(df.groupby('D')['ps'].describe()[['mean', 'min', 'max']])

## 3 · Overlap and balance — *before* matching

Two design checks, done before we touch the outcome again.

**(a) Overlap.** Plot the PS distribution in each group. Where they don't overlap, there are no comparable units.

**(b) Balance.** The standardized mean difference (SMD) per covariate.

In [ ]:
fig, ax = plt.subplots()
ax.hist(df.loc[df.D==1, 'ps'], bins=30, alpha=0.6, density=True,
        label='treated')
ax.hist(df.loc[df.D==0, 'ps'], bins=30, alpha=0.6, density=True,
        label='control')
ax.set_xlabel('propensity score  e(X)'); ax.set_ylabel('density')
ax.set_title('Overlap of the propensity score by group')
ax.legend()
# Treated mass sits to the right — but the supports do overlap.
print('PS overlap region:',
      f"[{max(df[df.D==1].ps.min(), df[df.D==0].ps.min()):.3f},",
      f"{min(df[df.D==1].ps.max(), df[df.D==0].ps.max()):.3f}]")

In [ ]:
def smd(a, b):
    """Standardized mean difference: mean gap / pooled SD."""
    return (a.mean() - b.mean()) / np.sqrt(
        (a.var(ddof=1) + b.var(ddof=1)) / 2)

trt = df[df.D == 1]; ctl = df[df.D == 0]
smd_before = {c: smd(trt[c], ctl[c]) for c in covs}
print('SMD before matching (|SMD| < 0.1 is the goal):')
for c in covs:
    print(f'  {c:6s} {smd_before[c]:+.3f}')
assert max(abs(v) for v in smd_before.values()) > 0.3, \
    'covariates should be clearly imbalanced before matching'

## 4 · 1:1 nearest-neighbor matching on the PS → the ATT

Match each treated unit to the control with the closest propensity score (`NearestNeighbors`). This targets the **ATT**. The ATT is then the mean within-pair outcome difference — and it should land near `2.0`.

In [ ]:
trt = df[df.D == 1].copy()
ctl = df[df.D == 0].copy()
nn = NearestNeighbors(n_neighbors=1).fit(ctl[['ps']].values)
_, idx = nn.kneighbors(trt[['ps']].values)
matched_ctl = ctl.iloc[idx.ravel()].copy()

att_match = (trt['Y'].values - matched_ctl['Y'].values).mean()
print(f'matched ATT = {att_match:.3f}')
print(f'true ATT    = {TRUE_ATT:.3f}')
print(f'naive       = {naive:.3f}   (for contrast)')
assert abs(att_match - TRUE_ATT) < 0.15, \
    'matching should recover the true ATT within 0.15'

### Balance *after* matching (the love-plot idea)

Recompute the SMDs on the matched sample. They should collapse toward zero — that is the love-plot: every covariate marching from imbalanced to balanced.

In [ ]:
smd_after = {c: smd(trt[c], matched_ctl[c]) for c in covs}

y = np.arange(len(covs))
fig, ax = plt.subplots()
ax.scatter([abs(smd_before[c]) for c in covs], y, label='before', s=70)
ax.scatter([abs(smd_after[c])  for c in covs], y, label='after', s=70)
ax.axvline(0.1, ls='--', color='grey')   # |SMD| = 0.1 threshold
ax.set_yticks(y); ax.set_yticklabels(covs)
ax.set_xlabel('|standardized mean difference|')
ax.set_title('Love-plot: covariate balance before vs after matching')
ax.legend()

print('covariate   before    after')
for c in covs:
    print(f'  {c:6s} {smd_before[c]:+.3f}   {smd_after[c]:+.3f}')
assert max(abs(v) for v in smd_after.values()) < 0.15, \
    'matching should bring every SMD near zero'

## 5 · Trimming the non-overlap region

Restrict to the **common support** — the PS range where both groups appear — and re-estimate. Here overlap is already good, so trimming drops only a few units and barely moves the ATT; the point is the *mechanic* and that trimming changes which units (hence which population) you analyze.

In [ ]:
lo = max(trt['ps'].min(), ctl['ps'].min())
hi = min(trt['ps'].max(), ctl['ps'].max())
on_support = df[(df.ps >= lo) & (df.ps <= hi)].copy()
print(f'common support PS in [{lo:.3f}, {hi:.3f}]')
print(f'kept {len(on_support)} of {len(df)} units '
      f'({len(df) - len(on_support)} trimmed)')

t2 = on_support[on_support.D == 1]; c2 = on_support[on_support.D == 0]
nn2 = NearestNeighbors(n_neighbors=1).fit(c2[['ps']].values)
_, idx2 = nn2.kneighbors(t2[['ps']].values)
att_trim = (t2['Y'].values - c2.iloc[idx2.ravel()]['Y'].values).mean()
print(f'ATT after trimming = {att_trim:.3f}  (vs untrimmed {att_match:.3f})')
assert abs(att_trim - TRUE_ATT) < 0.2

### 🔧 Exercise 5.1 — caliper matching on the logit

A **caliper** refuses any match whose propensity-score distance is too large, dropping low-quality pairs. The standard caliper is `0.2 × SD(logit(ps))`.

Match each treated unit to its nearest control **on `logit_ps`**, then keep only pairs within the caliper, and estimate the ATT on the kept pairs. Fill in the `# TODO`s.

In [ ]:
# TODO: build a caliper match on the logit of the propensity score.
caliper = 0.2 * df['logit_ps'].std()
print(f'caliper = {caliper:.3f} (on the logit scale)')

# nn_l = NearestNeighbors(n_neighbors=1).fit(...)      # fit on ctl logit_ps
# dist, idx_l = nn_l.kneighbors(...)                   # query trt logit_ps
# keep = ...               # boolean: dist within the caliper
# att_caliper = ...        # mean within-pair diff over KEPT pairs
# print(att_caliper, 'matched', int(keep.sum()), 'of', len(trt))
att_caliper = ...   # placeholder so the notebook still runs

### ✅ Solution 5.1

In [ ]:
nn_l = NearestNeighbors(n_neighbors=1).fit(ctl[['logit_ps']].values)
dist, idx_l = nn_l.kneighbors(trt[['logit_ps']].values)
dist = dist.ravel(); idx_l = idx_l.ravel()
keep = dist <= caliper

matched_y = ctl.iloc[idx_l]['Y'].values
att_caliper = (trt['Y'].values[keep] - matched_y[keep]).mean()
print(f'caliper ATT = {att_caliper:.3f}   '
      f'(kept {int(keep.sum())} of {len(trt)} treated)')
print(f'true ATT    = {TRUE_ATT:.3f}')
assert abs(att_caliper - TRUE_ATT) < 0.2, 'caliper match should recover ~2.0'

### 🔧 Exercise 5.2 — a second estimator: IPW for the ATT

Matching is not the only way to use the propensity score. **Inverse-probability weighting** for the ATT keeps treated units at weight 1 and weights each control by `e(X) / (1 − e(X))`, so the reweighted controls mimic the treated group's covariate distribution.

Estimate the ATT as `mean(Y | treated) − weighted_mean(Y | control)` and check it also recovers `2.0`. (This previews Week 7.)

In [ ]:
# TODO: ATT via inverse-probability weighting.
# odds = df['ps'] / (1 - df['ps'])      # the control weights
# treated_mean = ...                    # plain mean of treated Y
# control_wmean = ...                   # weighted mean of control Y, weights=odds
# att_ipw = treated_mean - control_wmean
# print(att_ipw)
att_ipw = ...   # placeholder so the notebook still runs

### ✅ Solution 5.2

In [ ]:
odds = (df['ps'] / (1 - df['ps'])).values
is_t = df['D'].values == 1
treated_mean  = df['Y'].values[is_t].mean()
control_wmean = np.average(df['Y'].values[~is_t], weights=odds[~is_t])
att_ipw = treated_mean - control_wmean
print(f'IPW ATT  = {att_ipw:.3f}')
print(f'true ATT = {TRUE_ATT:.3f}')
assert abs(att_ipw - TRUE_ATT) < 0.2, 'IPW should also recover ~2.0'

### 🔧 Exercise 5.3 — naive SEs are too small

Here we *demonstrate* the inference warning. The honest way to get a matching SE is to bootstrap the **whole pipeline**: resample the data, refit the PS, rematch, re-estimate. Compare the spread of those bootstrap ATTs to a naive within-pair SE.

Implement the full-pipeline bootstrap below.

In [ ]:
# TODO: full-pipeline bootstrap of the ATT.
def matched_att(data):
    """Refit PS, 1:1 match, return ATT for a dataframe."""
    Xb = data[covs].values
    psb = LogisticRegression(max_iter=1000).fit(Xb, data['D'].values)\
             .predict_proba(Xb)[:, 1]
    d = data.assign(ps=psb)
    t, c = d[d.D == 1], d[d.D == 0]
    j = NearestNeighbors(n_neighbors=1).fit(c[['ps']].values)\
          .kneighbors(t[['ps']].values, return_distance=False).ravel()
    return (t['Y'].values - c.iloc[j]['Y'].values).mean()

B = 200
# boot = np.array([matched_att(df.sample(len(df), replace=True,
#                 random_state=int(RNG.integers(1e9)))) for _ in range(B)])
# boot_se = boot.std(ddof=1)
boot_se = ...   # placeholder so the notebook still runs

### ✅ Solution 5.3

In [ ]:
B = 200
boot = np.array([
    matched_att(df.sample(len(df), replace=True,
                          random_state=int(RNG.integers(1_000_000_000))))
    for _ in range(B)])
boot_se = boot.std(ddof=1)

# A naive within-pair SE treats matched pairs as i.i.d. observations.
pair_diff = trt['Y'].values - matched_ctl['Y'].values
naive_se = pair_diff.std(ddof=1) / np.sqrt(len(pair_diff))

print(f'bootstrap (full-pipeline) SE = {boot_se:.3f}')
print(f'naive within-pair SE         = {naive_se:.3f}')
print(f'ratio = {boot_se / naive_se:.2f}x')
assert boot_se > naive_se, \
    'the naive SE understates uncertainty vs the full-pipeline bootstrap'

## 6 · Wrap-up & self-check

- The naive treated−control difference was badly biased (confounding).
- The **propensity score** `e(X)=P(D|X)` is a *balancing score*: matching on one number balanced all three covariates (SMDs → near 0).
- **1:1 NN matching on the PS** recovered the true ATT of `2.0`; so did **caliper matching** and **IPW** — three roads, one estimand.
- **Trimming** to common support enforces overlap but changes the population you analyze.
- **Naive post-matching SEs are too small**: the full-pipeline bootstrap gave a larger, honest SE.

**You're ready for Week 7** if you can state the balancing property, read a love-plot, and say why matching targets the ATT. Next week: weighting & doubly robust estimation — keep every unit, reweight the sample, and get a method that is right if *either* model is.